# RO-076 ROS 2: every number in the Note

The drive is RO-058's (`images/bandkit.py`, `images/trajkit.py`, copied, cached); `images/roskit.py` reads off what
the messages between the room robot's nodes would carry. No ROS installation is needed: the numbers are the contents
of the messages, computed from the drive. Nothing is random.

In [1]:
import sys, math
sys.path.insert(0, 'images')
import numpy as np
import roskit as rk
r2, r3, r4 = (lambda x: round(float(x), 2)), (lambda x: round(float(x), 3)), (lambda x: round(float(x), 4))

## 1. The messages around 2.4 s (Sections 1 and 2)

In [2]:
for t in (2.3, 2.4, 2.5):
    s = rk.scan(t); r = s['ranges']; b = np.degrees(s['bearings'])
    k = int(np.argmin(np.where(np.abs(b) < 30, r, 99)))
    print('/scan at', t, 's:', len(r), 'readings from', r2(math.degrees(s['angle_min'])), 'to', r2(math.degrees(s['angle_max'])),
          'deg in', r4(math.degrees(s['angle_increment'])), 'deg steps; nearest within 30 deg of ahead', r3(r[k]), 'm at', r2(b[k]), 'deg, index', k)
for t in (2.30, 2.35, 2.40, 2.45, 2.50):
    v, w = rk.cmd(t)
    print('/cmd_vel at', t, 's: linear.x', r3(v), 'm/s, angular.z', r3(w), 'rad/s; wheels right', r3(v + w * 0.1), 'left', r3(v - w * 0.1))
o = rk.L[24]
print('/local_plan at 2.4 s:', len(o['P']), 'poses')
print('messages per second: /scan', 10, '/odom', 20, '/local_plan', 10, '/cmd_vel', 20, '; total', 10 + 20 + 10 + 20)
print('one LaserScan: 682 ranges x 4 bytes =', 682 * 4, 'bytes, plus 7 numbers of 4 bytes and a header; 10 a second:', 682 * 4 * 10, 'bytes/s')
print('one Twist: 6 numbers x 8 bytes =', 6 * 8, 'bytes')

/scan at 2.3 s: 682 readings from -119.86 to 119.86 deg in 0.352 deg steps; nearest within 30 deg of ahead 1.202 m at -29.74 deg, index 256
/scan at 2.4 s: 682 readings from -119.86 to 119.86 deg in 0.352 deg steps; nearest within 30 deg of ahead 0.715 m at -6.51 deg, index 322
/scan at 2.5 s: 682 readings from -119.86 to 119.86 deg in 0.352 deg steps; nearest within 30 deg of ahead 0.659 m at -8.98 deg, index 315
/cmd_vel at 2.3 s: linear.x 0.586 m/s, angular.z 0.142 rad/s; wheels right 0.6 left 0.572
/cmd_vel at 2.35 s: linear.x 0.595 m/s, angular.z 0.054 rad/s; wheels right 0.6 left 0.589
/cmd_vel at 2.4 s: linear.x 0.592 m/s, angular.z 0.079 rad/s; wheels right 0.6 left 0.584
/cmd_vel at 2.45 s: linear.x 0.567 m/s, angular.z 0.329 rad/s; wheels right 0.6 left 0.534
/cmd_vel at 2.5 s: linear.x 0.542 m/s, angular.z 0.579 rad/s; wheels right 0.6 left 0.484
/local_plan at 2.4 s: 10 poses
messages per second: /scan 10 /odom 20 /local_plan 10 /cmd_vel 20 ; total 60
one LaserScan: 682 ran

## 2. The map service (Section 3)

In [3]:
nx, ny = rk.MAP_CELLS
print('map', rk.MAP_W, 'x', rk.MAP_H, 'm at', rk.MAP_RES, 'm cells:', nx, 'x', ny, '=', nx * ny, 'cells, one byte each')

map 7.0 x 4.0 m at 0.05 m cells: 140 x 80 = 11200 cells, one byte each


## 3. The navigate-to-pose action (Section 4)

In [4]:
print('plan from the robot to the door pose:', r3(rk.PLAN_LENGTH), 'm; arrival', rk.T_ARRIVE, 's')
for t in [0, 1, 2, 2.4, 3, 4, 5, 6, 7, 8]:
    f = rk.feedback(float(t))
    print('t', t, 's: distance remaining', r3(f['distance_remaining']), 'm; speed', r3(f['speed']), 'm/s; estimated time remaining',
          r2(f['estimated_time_remaining']), 's')
print('feedback every 10 ms for 8.0 s:', int(8.0 / 0.01), 'messages')

plan from the robot to the door pose: 3.698 m; arrival 8.0 s
t 0 s: distance remaining 3.698 m; speed 0.0 m/s; estimated time remaining 0.0 s
t 1 s: distance remaining 3.454 m; speed 0.5 m/s; estimated time remaining 6.91 s
t 2 s: distance remaining 2.908 m; speed 0.578 m/s; estimated time remaining 5.03 s
t 2.4 s: distance remaining 2.457 m; speed 0.595 m/s; estimated time remaining 4.13 s
t 3 s: distance remaining 2.172 m; speed 0.489 m/s; estimated time remaining 4.44 s
t 4 s: distance remaining 1.707 m; speed 0.461 m/s; estimated time remaining 3.7 s
t 5 s: distance remaining 1.235 m; speed 0.507 m/s; estimated time remaining 2.44 s
t 6 s: distance remaining 0.932 m; speed 0.569 m/s; estimated time remaining 1.64 s
t 7 s: distance remaining 0.092 m; speed 0.411 m/s; estimated time remaining 0.0 s
t 8 s: distance remaining 0.004 m; speed 0.0 m/s; estimated time remaining 0.0 s
feedback every 10 ms for 8.0 s: 800 messages


## 4. Quality of service (Section 7)

In [5]:
# scans every 0.1 s from 2.2 s; the 2.4 s scan is lost on the link; with reliable delivery its copy is assumed to
# arrive 0.15 s late, at 2.55 s, and the 2.5 s scan is held back until then (messages keep their order)
sent = [2.2, 2.3, 2.4, 2.5, 2.6, 2.7]
best = {t: t for t in sent if t != 2.4}
rel = {}
for t in sent:
    arr = 2.55 if t == 2.4 else t
    rel[t] = max([arr] + [rel[u] for u in rel])
print('best effort, arrival times:', best)
print('reliable, arrival times:', rel)
def newest(arr, now):
    got = [t for t, a in arr.items() if a <= now + 1e-9]
    return max(got) if got else None
for now in (2.4, 2.5, 2.55):
    print('at', now, 's the newest scan: best effort', newest(best, now), '; reliable', newest(rel, now))

best effort, arrival times: {2.2: 2.2, 2.3: 2.3, 2.5: 2.5, 2.6: 2.6, 2.7: 2.7}
reliable, arrival times: {2.2: 2.2, 2.3: 2.3, 2.4: 2.55, 2.5: 2.55, 2.6: 2.6, 2.7: 2.7}
at 2.4 s the newest scan: best effort 2.3 ; reliable 2.3
at 2.5 s the newest scan: best effort 2.5 ; reliable 2.3
at 2.55 s the newest scan: best effort 2.5 ; reliable 2.5


In [6]:
# history and depth: the planner is busy from 2.4 s to 2.7 s; scans arrive every 0.1 s
waiting = [2.5, 2.6, 2.7]
for name, depth in (('keep last, depth 1', 1), ('keep all', None)):
    q = waiting[-depth:] if depth else waiting
    print(name, ': kept', q, '; read first at 2.7 s:', q[0], ', age', round(2.7 - q[0], 1), 's')

keep last, depth 1 : kept [2.7] ; read first at 2.7 s: 2.7 , age 0.0 s
keep all : kept [2.5, 2.6, 2.7] ; read first at 2.7 s: 2.5 , age 0.2 s


## 5. Lifecycle start-up order (Section 9)

In [7]:
# Nav2 main, bringup_launch.py: localization nodes, then the keepout-zone and speed-zone filter servers (both switched
# on by default), then the 11 servers of navigation_launch.py
nodes = ['map_server', 'amcl', 'keepout_filter_mask_server', 'keepout_costmap_filter_info_server',
         'speed_filter_mask_server', 'speed_costmap_filter_info_server', 'controller_server', 'smoother_server', 'planner_server', 'route_server', 'behavior_server',
         'velocity_smoother', 'collision_monitor', 'bt_navigator', 'waypoint_follower', 'docking_server', 'following_server']
print(len(nodes), 'nodes managed; start-up: configure all in this order, then activate all in this order; shut-down in reverse')
print('bond: heartbeat every 0.25 s, a node is declared lost after 4.0 s without one:', int(4.0 / 0.25), 'missed heartbeats')

17 nodes managed; start-up: configure all in this order, then activate all in this order; shut-down in reverse
bond: heartbeat every 0.25 s, a node is declared lost after 4.0 s without one: 16 missed heartbeats
